# Lesson 12 - Chat History Reduction wit Agent Scratchpad

Dis notebook dey show how to manage context for long conversations using Microsoft Agent Framework. As conversation dem dey grow, token count go increase — e go finally pass di model context window. We go solve dis wit **context summarization pattern** and **agent scratchpad** wey get persistent memory.

## Wetin You Go Learn:
1. **Why Context Management Matter**: Understanding token limits and context windows
2. **Context-Aware Agents**: How to build agents wey go manage dia own conversation context
3. **Context Summarization Pattern**: How to use tools take condense conversation history
4. **Agent Scratchpad**: Persistent memory wey dey survive context reduction

## Prerequisites:
- Azure OpenAI setup wit environment variables configured
- Understanding of basic agent concepts from previous lessons


## Setup


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Why Context Management Matters

Every LLM get limited **context window** — na the maximum number of tokens e fit process for one single request. As multi-turn conversation dey go on:

- **Token count dey grow linearly** with each user message and assistant reply.
- **Prompt tokens dey cost pass** because the whole history dey sent again every turn.
- At last the conversation **go pass the context window** and the model go either cut am short or give error.

### Strategies for Managing Context

| Strategy | How E Dey Work | Trade-off |
|---|---|---|
| **Truncation** | Drop oldest messages | Go lose early context |
| **Summarization** | Condense older messages into summary | Some detail go lost, but key points go remain |
| **Scratchpad / External Memory** | Store key facts outside the conversation | Need tool calls, but e go still dey even if them reduce am |

For this notebook we combine **summarization** with **scratchpad tool** so the agent fit still maintain continuity even if conversation history shrink.


## Di Agent Wey Dey Understand Context


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Simulating a Long Conversation

Make we waka through one long talk wey get many turn dem make we see how context dey build up. The agent suppose hold key details (preferences, budget, travel dates) across the turns and show say e fit continue well well.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Notice how the agent dey keep context from earlier turns — e sabi about Japan, sushi, temples, photography, the $3000 budget, solo travel, and the April timeframe. For one short conversation, dis one dey work well, but as the conversation dey grow, to keep all the history dey expensive to resend.

Make we continue the conversation with more turns to see how context dey accumulate:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Context Summarization Pattern

As di tok-tok dey grow, we fit use **summarization tool** to condense accumulated context into small-small format. Di agent dey call dis tool to record key preferences so dat even if old messages dem drop, di important information go still dey.

Dis pattern na di building block for more advanced history reduction:
1. Di agent go find key facts from di conversation
2. E go call di summarization tool to keep dem
3. Old messages fit comot without wahala because di summary dey capture wetin important

Below, we define one `summarize_preferences` tool wey di agent fit call to record one small summary of wetin e don learn.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Summary

For dis lesson, you learn how to manage context for long-running agent conversations using Microsoft Agent Framework:

### Key Concepts
- **Context windows get limit** — every token wey dey inside conversation history dey cost money and count towards the limit.
- **Summarization tools** dey make the agent fit shrink accumulated context into small summaries, reduce token usage but still keep important info.
- **Agent scratchpads** dey give permanent external memory wey go still dey after any conversation reduction.

### Wetin You Build
- One **context-aware agent** wey dey maintain conversation continuity across multi-turn talks
- One **summarization tool** (`summarize_preferences`) wey dey record key user details for one small format
- One **multi-turn conversation** wey show how context fit stay and how to handle changes

### Real-World Applications
- **Customer Service Bots**: Dem fit remember preferences for long support sessions
- **Personal Assistants**: Dem fit track ongoing projects without make person talk context again
- **Educational Tutors**: Dem fit maintain student progress across many talks

### Next Steps
- Make full scratchpad tool wey get file-based persistence
- Add automatic history truncation after summarization
- Join with vector databases for semantic memory search
- Build agents wey fit resume conversation later with full context


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:  
Dis document na di one wey AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator) translate am. Even tho we dey try make am correct, abeg sabi say automated translation fit get some error or mistake. Di original document wey dey dia in e own language na im be di correct one. If na serious info, e better make human professional translate am. We no go responsible if person buy mistake or no understand well because of dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
